In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# LLM 선언

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini") 
small_llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
# 도구 선언
# 데코레이터 import
from langchain_core.tools import tool

# expected arguments
# tool name
# description

@tool
def add(a: int, b:int) -> int:
    """
        숫자 a 와 b를 더합니다    
    """
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """
        숫자 a 와 b를 곱합니다    
    """
    return a * b



# def multiply(a: int, b: int) -> int:
#     """
#         숫자 a 와 b를 곱합니다    
#     """
#     return a * b



In [4]:
# multiply(4, 8)

In [5]:
# add(4, 8)  # TypeError: 'StructuredTool' object is not callable 
# 함수처럼 호출하면 에러

In [6]:
add.invoke({'a':4, 'b':8})  # invoke해서 실행, 인자를 딕셔너리로 넘겨야 한다

12

##### LLM이 도구를 활용할 수 있도록 하는 방법

In [7]:
llm_with_tools = small_llm.bind_tools([add, multiply])  # 도구들을 리스트 형태로 할당
# llm 인스턴스 -> bind_tools 메서드가 있음
# bing_functions 보다는 bind_tools 권장

In [8]:
# llm이 도구를 호출해줘야 함.

query  = '3 곱하기 5는?'

In [9]:
# 그냥 LLM을 호출하면 AIMessage 의 content에 답이 담긴다
small_llm.invoke(query)  

AIMessage(content='3 곱하기 5는 15입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 15, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a1681c17ec', 'id': 'chatcmpl-DKcU8HTQDifmvsFZhYnDFNSMd1xer', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cff20-d78e-7ee1-9d3c-99f9317b3d29-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 11, 'total_tokens': 26, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [10]:
# additional_kwargs 에 'tool_calls'라는 키값이 생긴다

result = llm_with_tools.invoke(query)  

In [11]:
result.tool_calls  #  호출할 도구들이 list로 담긴다. 
# 지금은 곱셈만 하면 되므로, multiply만 호출

[{'name': 'multiply',
  'args': {'a': 3, 'b': 5},
  'id': 'call_lLigdC2V5Ze7UE5p3Asn4YTU',
  'type': 'tool_call'}]

In [12]:
# 결론

# 결국, LLM에게 도구를 던져주는 것 -> LLM이 도구를 invoke할 수 있게 해줘야 한다

In [13]:
# How? 
# state에 "messages : sequence[AnyMessage]" 를 선언했었음.
# AnyMessage에는 
    # -SystemMesage -> 시스템 프롬프트 (디벨로퍼)
    # -Humanmessage -> 유저가 작성하는
    # -AiMessage(AI가 생성하는) + ToolMessage가 생긴다 

# LLM이 ToolMessage들을 호출할 수 있게 해줘야 함


# how? LLM에서 히스토리 관리를 할때 리스트로 넘겨줬었다. 
    # ('human', '...')
    # ('ai', '...') 이런식....

# 위 방식처럼 넘겨줘서 invoke를 해줘야 LLM이 도구를 사용할 수 있게 된다.
# 이유는, 도구만 주면 실행이 안되기 때문

# invoke(질문) -> 이 방식이 아니라, 
# invoke (- human) -> 를 해줘야 LLM이 정상적으로 답변을 생성하게 된다. 
#        (- ai)
#        (- tool)

In [14]:
from langchain_core.messages import AnyMessage, HumanMessage
from typing import Sequence

human_message = HumanMessage(query)  # Human Message 선언
message_list : Sequence[AnyMessage] = [human_message]  # human message 추가


# (type) AnyMessage = AIMessage | HumanMessage | ChatMessage | SystemMessage | FunctionMessage | ToolMessage | AIMessageChunk | HumanMessageChunk | ChatMessageChunk | SystemMessageChunk | FunctionMessageChunk | ToolMessageChunk

In [15]:
ai_message = llm_with_tools.invoke(message_list)

In [16]:
ai_message.tool_calls # 도구 호출 결과

[{'name': 'multiply',
  'args': {'a': 3, 'b': 5},
  'id': 'call_Ru5Eu4bOzTdPrfWA4PNxSI4T',
  'type': 'tool_call'}]

In [17]:
message_list.append(ai_message)  # message_list 에 도구 호출 결과를 추가. 

In [ ]:
# 도구의 호출을 ai_message로 한다

tool_message = multiply.invoke(ai_message.tool_calls[0])

In [19]:
message_list.append(tool_message) # tool_message도 append

In [20]:
llm_with_tools.invoke(message_list)  # 여기서 invoke를 해준다. 

AIMessage(content='3 곱하기 5는 15입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 109, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ca3e7d71bf', 'id': 'chatcmpl-DKcUED1LymRwSMVT0Z4DZOESfkZTV', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cff20-f113-7c10-a723-ae7d3fbaaae8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 109, 'output_tokens': 12, 'total_tokens': 121, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})